# Pratique - IBGE e Banco Central

## Extração e tratamento de dados públicos

Este notebook tem como objetivo realizar a extração de dados públicos do
Banco Central do Brasil utilizando a biblioteca `requests`, transformar
os dados obtidos em um DataFrame com `pandas`, realizar tratamentos,
validar os resultados e salvar os dados em diferentes formatos.

## 1. Importação das bibliotecas

Nesta etapa são importadas as bibliotecas necessárias para o projeto:

- `requests`: utilizada para realizar requisições HTTP à API do Banco Central.
- `pandas`: utilizada para organizar, analisar e tratar os dados obtidos.

In [1]:
import requests
import pandas as pd

## 2. Requisição à API do Banco Central

Nesta etapa será realizada uma requisição HTTP à API pública do Banco Central do Brasil para obter dados de transações Pix por município.

In [2]:
url = (
    "https://olinda.bcb.gov.br/olinda/servico/Pix_DadosAbertos/versao/v1/odata/"
    "TransacoesPixPorMunicipio(DataBase='202501')"
)

params = {
    "formato": "json",
    "$top": 100000
}

In [3]:
response = requests.get(url, params=params, timeout=30)

response.raise_for_status()

In [4]:
response.status_code

200

## 3. Conversão da resposta para DataFrame

A resposta da API é recebida em formato JSON. Nesta etapa, os registros serão extraídos e convertidos para um DataFrame utilizando a biblioteca pandas.

In [5]:
dados_json = response.json()

dados_json.keys()

dict_keys(['@odata.context', 'value'])

In [6]:
dados = dados_json["value"]

df = pd.DataFrame(dados)

In [7]:
df.head()

,AnoMes,Municipio_Ibge,Municipio,Estado_Ibge,Estado,Sigla_Regiao,Regiao,VL_PagadorPF,QT_PagadorPF,VL_PagadorPJ,QT_PagadorPJ,VL_RecebedorPF,QT_RecebedorPF,VL_RecebedorPJ,QT_RecebedorPJ,QT_PES_PagadorPF,QT_PES_PagadorPJ,QT_PES_RecebedorPF,QT_PES_RecebedorPJ
0,202504,3201803.0,DIVINO DE SÃO LOURENÇO,32.0,ESPÍRITO SANTO,SE,SUDESTE,1.305785e+07,75070,6126647.40,4623,1.099215e+07,38063,4312249.56,16196,2477,186,2403,189
1,202607,3137502.0,LAGOA FORMOSA,31.0,MINAS GERAIS,SE,SUDESTE,1.315881e+08,335115,49388535.11,23739,1.193842e+08,188118,60958314.67,80441,10505,866,10160,818
2,202505,2600401.0,ÁGUA PRETA,26.0,PERNAMBUCO,NE,NORDESTE,6.628262e+07,578555,5345807.76,7828,6.273276e+07,367600,5828835.69,40424,14861,253,14822,226
3,202510,4310900.0,JACUTINGA,43.0,RIO GRANDE DO SUL,SU,SUL,9.603779e+06,34582,5347854.04,2420,7.669269e+06,16739,4586782.05,6739,1816,193,1567,187
4,202603,5005152.0,JUTI,50.0,MATO GROSSO DO SUL,CO,CENTRO-OESTE,2.462597e+07,155984,10083116.18,11082,2.380106e+07,76860,9785377.48,31407,4143,289,4006,272


In [8]:
df.shape

(100000, 19)

## 4. Salvamento dos dados brutos

Após a conversão da resposta da API para DataFrame, os dados brutos serão salvos em um arquivo CSV para preservar o resultado original da extração.

In [9]:
df.to_csv(
    "../data/raw/bcb_tabela.csv",
    index=False,
    encoding="utf-8-sig"
)

In [10]:
print("Arquivo bcb_tabela.csv salvo com sucesso.")

Arquivo bcb_tabela.csv salvo com sucesso.


## 5. Análise inicial dos dados

Antes de realizar os tratamentos, será feita uma verificação da estrutura dos dados, dos tipos das colunas, da presença de valores nulos e de registros duplicados.

In [11]:
df.dtypes

AnoMes                  int64
Municipio_Ibge        float64
Municipio                 str
Estado_Ibge           float64
Estado                    str
Sigla_Regiao              str
Regiao                    str
VL_PagadorPF          float64
QT_PagadorPF            int64
VL_PagadorPJ          float64
QT_PagadorPJ            int64
VL_RecebedorPF        float64
QT_RecebedorPF          int64
VL_RecebedorPJ        float64
QT_RecebedorPJ          int64
QT_PES_PagadorPF        int64
QT_PES_PagadorPJ        int64
QT_PES_RecebedorPF      int64
QT_PES_RecebedorPJ      int64
dtype: object

In [12]:
df.isnull().sum()

AnoMes                 0
Municipio_Ibge        22
Municipio              0
Estado_Ibge           22
Estado                 0
Sigla_Regiao          22
Regiao                 0
VL_PagadorPF           0
QT_PagadorPF           0
VL_PagadorPJ           0
QT_PagadorPJ           0
VL_RecebedorPF         0
QT_RecebedorPF         0
VL_RecebedorPJ         0
QT_RecebedorPJ         0
QT_PES_PagadorPF       0
QT_PES_PagadorPJ       0
QT_PES_RecebedorPF     0
QT_PES_RecebedorPJ     0
dtype: int64

In [13]:
df.duplicated().sum()

np.int64(0)

## 6. Tratamento dos dados

Serão realizados três tratamentos:

1. Remoção de espaços extras dos nomes das colunas.
2. Remoção de espaços extras dos valores das colunas de texto.
3. Conversão dos códigos IBGE para o tipo inteiro.

In [14]:
df.columns = df.columns.str.strip()

df.columns

Index(['AnoMes', 'Municipio_Ibge', 'Municipio', 'Estado_Ibge', 'Estado',
       'Sigla_Regiao', 'Regiao', 'VL_PagadorPF', 'QT_PagadorPF',
       'VL_PagadorPJ', 'QT_PagadorPJ', 'VL_RecebedorPF', 'QT_RecebedorPF',
       'VL_RecebedorPJ', 'QT_RecebedorPJ', 'QT_PES_PagadorPF',
       'QT_PES_PagadorPJ', 'QT_PES_RecebedorPF', 'QT_PES_RecebedorPJ'],
      dtype='str')

In [15]:
colunas_texto = df.select_dtypes(include="str").columns

for coluna in colunas_texto:
    df[coluna] = df[coluna].str.strip()

In [16]:
df["Municipio_Ibge"] = df["Municipio_Ibge"].astype("Int64")
df["Estado_Ibge"] = df["Estado_Ibge"].astype("Int64")

## 7. Validação dos tratamentos

Após os tratamentos, serão realizadas novas verificações para confirmar a qualidade dos dados.

In [17]:
df.dtypes

AnoMes                  int64
Municipio_Ibge          Int64
Municipio                 str
Estado_Ibge             Int64
Estado                    str
Sigla_Regiao              str
Regiao                    str
VL_PagadorPF          float64
QT_PagadorPF            int64
VL_PagadorPJ          float64
QT_PagadorPJ            int64
VL_RecebedorPF        float64
QT_RecebedorPF          int64
VL_RecebedorPJ        float64
QT_RecebedorPJ          int64
QT_PES_PagadorPF        int64
QT_PES_PagadorPJ        int64
QT_PES_RecebedorPF      int64
QT_PES_RecebedorPJ      int64
dtype: object

In [18]:
df.isnull().sum()

AnoMes                 0
Municipio_Ibge        22
Municipio              0
Estado_Ibge           22
Estado                 0
Sigla_Regiao          22
Regiao                 0
VL_PagadorPF           0
QT_PagadorPF           0
VL_PagadorPJ           0
QT_PagadorPJ           0
VL_RecebedorPF         0
QT_RecebedorPF         0
VL_RecebedorPJ         0
QT_RecebedorPJ         0
QT_PES_PagadorPF       0
QT_PES_PagadorPJ       0
QT_PES_RecebedorPF     0
QT_PES_RecebedorPJ     0
dtype: int64

In [19]:
df.duplicated().sum()

np.int64(0)

In [20]:
df.head()

,AnoMes,Municipio_Ibge,Municipio,Estado_Ibge,Estado,Sigla_Regiao,Regiao,VL_PagadorPF,QT_PagadorPF,VL_PagadorPJ,QT_PagadorPJ,VL_RecebedorPF,QT_RecebedorPF,VL_RecebedorPJ,QT_RecebedorPJ,QT_PES_PagadorPF,QT_PES_PagadorPJ,QT_PES_RecebedorPF,QT_PES_RecebedorPJ
0,202504,3201803,DIVINO DE SÃO LOURENÇO,32,ESPÍRITO SANTO,SE,SUDESTE,1.305785e+07,75070,6126647.40,4623,1.099215e+07,38063,4312249.56,16196,2477,186,2403,189
1,202607,3137502,LAGOA FORMOSA,31,MINAS GERAIS,SE,SUDESTE,1.315881e+08,335115,49388535.11,23739,1.193842e+08,188118,60958314.67,80441,10505,866,10160,818
2,202505,2600401,ÁGUA PRETA,26,PERNAMBUCO,NE,NORDESTE,6.628262e+07,578555,5345807.76,7828,6.273276e+07,367600,5828835.69,40424,14861,253,14822,226
3,202510,4310900,JACUTINGA,43,RIO GRANDE DO SUL,SU,SUL,9.603779e+06,34582,5347854.04,2420,7.669269e+06,16739,4586782.05,6739,1816,193,1567,187
4,202603,5005152,JUTI,50,MATO GROSSO DO SUL,CO,CENTRO-OESTE,2.462597e+07,155984,10083116.18,11082,2.380106e+07,76860,9785377.48,31407,4143,289,4006,272


### Resultado da validação

Após os tratamentos, foi verificado que:

- não foram encontrados registros duplicados;
- foram identificados 22 valores ausentes nas colunas `Municipio_Ibge`, `Estado_Ibge` e `Sigla_Regiao`;
- os códigos IBGE foram ajustados para o tipo inteiro;
- os espaços extras dos nomes das colunas e dos campos de texto foram removidos.

Os valores ausentes foram mantidos, pois não foi possível determinar, apenas pela análise inicial, se sua remoção seria adequada.

## 8. Salvamento dos dados tratados

Após a limpeza e validação, o conjunto de dados tratado será salvo nos formatos CSV e Parquet.

In [21]:
df.to_csv(
    "../data/processed/dados_tratados.csv",
    index=False,
    encoding="utf-8-sig"
)

df.to_parquet(
    "../data/processed/dados_tratados.parquet",
    index=False
)

print("Arquivos dados_tratados.csv e dados_tratados.parquet salvos com sucesso.")

Arquivos dados_tratados.csv e dados_tratados.parquet salvos com sucesso.


In [22]:
teste_parquet = pd.read_parquet(
    "../data/processed/dados_tratados.parquet"
)

teste_parquet.head()

,AnoMes,Municipio_Ibge,Municipio,Estado_Ibge,Estado,Sigla_Regiao,Regiao,VL_PagadorPF,QT_PagadorPF,VL_PagadorPJ,QT_PagadorPJ,VL_RecebedorPF,QT_RecebedorPF,VL_RecebedorPJ,QT_RecebedorPJ,QT_PES_PagadorPF,QT_PES_PagadorPJ,QT_PES_RecebedorPF,QT_PES_RecebedorPJ
0,202504,3201803,DIVINO DE SÃO LOURENÇO,32,ESPÍRITO SANTO,SE,SUDESTE,1.305785e+07,75070,6126647.40,4623,1.099215e+07,38063,4312249.56,16196,2477,186,2403,189
1,202607,3137502,LAGOA FORMOSA,31,MINAS GERAIS,SE,SUDESTE,1.315881e+08,335115,49388535.11,23739,1.193842e+08,188118,60958314.67,80441,10505,866,10160,818
2,202505,2600401,ÁGUA PRETA,26,PERNAMBUCO,NE,NORDESTE,6.628262e+07,578555,5345807.76,7828,6.273276e+07,367600,5828835.69,40424,14861,253,14822,226
3,202510,4310900,JACUTINGA,43,RIO GRANDE DO SUL,SU,SUL,9.603779e+06,34582,5347854.04,2420,7.669269e+06,16739,4586782.05,6739,1816,193,1567,187
4,202603,5005152,JUTI,50,MATO GROSSO DO SUL,CO,CENTRO-OESTE,2.462597e+07,155984,10083116.18,11082,2.380106e+07,76860,9785377.48,31407,4143,289,4006,272


In [23]:
teste_parquet.shape

(100000, 19)

## 9. Questões teóricas

### Qual a importância do web scraping no ambiente de trabalho?

O web scraping permite coletar informações disponíveis na internet de forma automatizada. No ambiente de trabalho, pode ser utilizado para pesquisas, acompanhamento de preços, levantamento de dados públicos, criação de bases de dados e apoio à tomada de decisões.

### Quais são os principais riscos legais e éticos envolvidos?

A coleta de dados deve respeitar os termos de uso das fontes, os direitos autorais e a legislação relacionada à proteção de dados pessoais, como a LGPD.

Entre os principais riscos estão:

- coletar dados contrariando os termos de uso de uma página;
- utilizar conteúdo protegido por direitos autorais de maneira inadequada;
- coletar ou divulgar dados pessoais sem necessidade ou finalidade adequada;
- realizar muitas requisições e prejudicar o funcionamento de um site.

Duas medidas práticas para reduzir esses riscos são:

1. Consultar os termos de uso e as regras de acesso da fonte antes da coleta.
2. Evitar a coleta de dados pessoais desnecessários e realizar anonimização quando necessário.

### Qual a diferença entre uma requisição via API e uma requisição sem API?

Uma API disponibiliza os dados de maneira estruturada e preparada para ser acessada por outros sistemas, normalmente em formatos como JSON ou XML.

Em uma coleta sem API, os dados geralmente precisam ser extraídos diretamente do conteúdo HTML de uma página por meio de técnicas de web scraping.

Sempre que existir uma API oficial disponível, ela é geralmente a abordagem mais adequada, pois oferece dados estruturados, facilita o tratamento das informações e reduz a dependência da estrutura visual da página, que pode sofrer alterações.